# EmPath normalized inputs and learned route preferences

**Purpose:** inspect comparable input scales and an empirical geospatial ML baseline. The existing interactive map is unchanged. A learned model is separate from GIS normalization and separate from routing. This notebook does not claim local validation or AI-learned traffic weights.

In [1]:
from pathlib import Path
import sys,json
ROOT=Path.cwd()
if not (ROOT/'scripts/learn_route_preferences.py').exists():ROOT=ROOT.parent
assert (ROOT/'scripts/learn_route_preferences.py').exists()
sys.path.insert(0,str(ROOT/'scripts'))
print('Python:',sys.executable)


Python: /Users/nalinikantheti/Documents/ChatGPT/EmPath/.venv/bin/python


## 1 What is normalized?
- Canopy deficit = 1 − canopy percent/100. Higher means less cover.
- Traffic burden = City of Plano traffic index/100. Higher means more nearby traffic.
- Distance remains additive meters/kilometers; it is not min-max ranked per edge.
- NOAA weather is constant regional context. No spatial heat score is invented.

These scales do not determine relative importance. No arbitrary combined walking index has been assigned.

In [2]:
import pandas as pd
folder=ROOT/'output/friction_features'
if (folder/'normalized_edges.csv').exists():
    edges=pd.read_csv(folder/'normalized_edges.csv')
    display(edges.head())
    display(edges[['canopy_deficit_01','traffic_burden_01','distance_km']].describe())
    print('Complete environmental features:',int(edges.environment_features_complete.sum()),'of',len(edges))
else:
    print('Run scripts/prepare_friction_features.py first.')

,u,v,key,length,distance_km,baseline_walk_seconds,canopy_cover_01,canopy_deficit_01,traffic_burden_01,environment_features_complete,weather_is_spatially_discriminating
0,26453885,13103365094,0,13.663834,0.013635,11.362228,0.0,1.0,0.966413,True,False
1,26454012,1371470529,0,100.378688,0.100130,83.441422,0.0,1.0,0.961647,True,False
2,26454012,11752576347,0,9.341432,0.009318,7.765215,0.0,1.0,0.963926,True,False
3,26454012,11752576346,0,9.986257,0.009999,8.332794,0.0,1.0,0.965131,True,False
4,26454015,1371470516,0,100.141990,0.099894,83.244695,0.0,1.0,0.961969,True,False


,canopy_deficit_01,traffic_burden_01,distance_km
count,93262.000000,84678.000000,93452.000000
mean,0.927317,0.494763,0.066957
std,0.101746,0.342424,0.083873
min,0.050000,0.012280,0.000314
25%,0.871501,0.117172,0.017300
50%,0.972812,0.577628,0.041905
75%,1.000000,0.822444,0.082729
max,1.000000,0.994632,3.199907


Complete environmental features: 84488 of 93452


## 2 The learned component
A simplified pooled route-choice model learns how distance and sun-exposed distance relate to observed choices between two paths. It uses the public [Singapore study](https://doi.org/10.1038/s41598-022-06383-5) and [released data](https://osf.io/aj4vk/). This is not the paper's full hierarchical model.

Participant-grouped evaluation prevents one person's trials appearing on both sides of a split. No traffic, crossing or medical-safety effects can be learned from the provided labels.

In [3]:
path=ROOT/'output/route_preference_model/model_and_evaluation.json'
if path.exists():
    model=json.loads(path.read_text())
    cv=model['grouped_cross_validation']
    display(pd.DataFrame({k:v for k,v in cv.items() if k in ['distance_only','distance_and_sun']}).T)
    print('Observations:',model['observations'],'Participants:',model['participants'])
    print('Full-data extra sun-distance factor:',round(model['full_data_extra_sun_distance_factor'],3))
    print('Performance is roughly comparable to distance-only. No Plano validation claimed.')
else:
    print('Run scripts/learn_route_preferences.py first.')

,log_loss,brier_score,accuracy,observations
distance_only,0.581065,0.194079,0.732843,408.0
distance_and_sun,0.577136,0.193747,0.727941,408.0


Observations: 408 Participants: 46
Full-data extra sun-distance factor: 0.191
Performance is roughly comparable to distance-only. No Plano validation claimed.


## 3 How it could feed routing
`equivalent meters = length meters + learned extra sun factor × sun-exposed meters`

The model's relative sun penalty is learned from observed choices; the cost remains additive and nonnegative. **We do not have observed time-specific sun exposure along every Plano edge.** Substituting canopy deficit is a separate scenario assumption, not a learned fact. Traffic trade-offs remain uncalibrated.

The model has not been applied automatically to your graph.

In [4]:
from learn_route_preferences import equivalent_distance
if path.exists():
    penalty=model['full_data_extra_sun_distance_factor']
    # Illustrative routes only, not observations from Plano.
    display(pd.DataFrame({'route':['400 m, all sun','450 m, shaded'],
        'distance_m':[400,450],
        'illustrative_equivalent_m':equivalent_distance([400,450],[400,0],penalty)}))

,route,distance_m,illustrative_equivalent_m
0,"400 m, all sun",400,476.244036
1,"450 m, shaded",450,450.000000


## 4 Optional rebuild
Existing outputs are loaded by default. This does not install new ML dependencies or change the production map/graph.

In [5]:
REBUILD=False
if REBUILD:
    import os
    os.environ.pop('PROJ_DATA',None);os.environ.pop('GDAL_DATA',None)
    from prepare_friction_features import main as normalize_inputs
    from learn_route_preferences import main as fit_model
    normalize_inputs();fit_model()
else:
    print('Preview only. See docs/GEOAI_AND_FRICTION.md for interpretation and next steps.')

Preview only. See docs/GEOAI_AND_FRICTION.md for interpretation and next steps.
